# 09a | A/B testing: spoken review

A must in DS interviews (especially product / analytics roles), usually as an open-ended question: "We want to launch a new feature. How would you design the experiment?" The interviewer is looking for a **complete framework** and whether you know the common pitfalls.

About 45 minutes.

> **How to use this page:** Answer each question **out loud** first (or write it in your notes for this page), then open the reference answer and compare. Statistics is your home turf, so the point here is not whether you understand it but whether you can explain it clearly in 30–60 seconds. Each reference answer ends with a **Short version**: the spoken answer in a few sentences. Practice it until it comes out naturally.

## 1. Why randomization matters

Random assignment makes "which group you're in" independent of every user characteristic (including the ones we can't observe), so the two groups are comparable before the experiment and any difference between them can be attributed to the treatment itself. That is why randomized experiments support causal conclusions, which observational data struggle to do.

## 2. The full experiment design process

Memorize these 10 steps and walk through them when answering a design question:

1. **Goal and hypothesis:** What is the change meant to improve? What decision will be made?
2. **Primary metric:** one, decided in advance;
3. **Guardrail metrics:** things that must not get worse, such as crash rate, load time, unsubscribe rate, revenue;
4. **Randomization unit:** usually the user, not the page view or session;
5. **Sample size and duration:** determined by the minimum detectable effect (MDE), α and power; run for at least 1–2 full weeks;
6. **Assignment and rollout:** start with a small ramp of traffic to make sure there are no bugs;
7. **Sanity checks:** does the sample split match expectations (SRM), are the logs healthy, A/A test;
8. **Analysis:** estimate the effect + confidence interval;
9. **Multiple testing and peeking:** handle them according to the pre-specified plan;
10. **Decision:** statistical significance + practical significance + guardrails + cost and risk.

**🗣 Say it out loud:** "We want to test a new recommendation ranking algorithm, with the goal of improving user retention." Walk through the complete experiment design.

<details>
<summary><b>Reference answer</b></summary>

**Goal:** does the new ranking improve retention, to decide whether to launch it to everyone.

**Metrics:** the primary metric is 7-day retention (28-day if time allows); since retention responds slowly, add a leading indicator such as sessions per user. Guardrails: crash rate, page load time, complaint / negative feedback rate, content diversity.

**Randomization unit:** the user, so the same user always sees the same version; if users interact with each other (social features, two-sided marketplaces), consider randomizing by cluster or region.

**Sample size:** from the baseline retention rate, the smallest lift worth detecting (say 1 percentage point absolute), α = 0.05 and power = 0.8, compute the users needed per group, then derive the duration from the number of new users entering the experiment each day; cover at least two full weeks to smooth out day-of-week and novelty effects.

**Checks:** is the split 50/50 (SRM), and are pre-experiment metrics the same in both groups.

**Analysis:** compare retention between groups at the user level and report the difference with a 95% confidence interval; state in advance that retention is the only confirmatory metric.

**Decision:** launch only if retention improves significantly, no guardrail worsens, and the lift is worth the maintenance cost. Also discuss whether the novelty effect fades, whether the effect differs between new and existing users, and long-term effects.

**Short version:** I'd define the decision and a single primary metric such as 7-day retention, plus guardrails like latency and crash rate. Randomize at the user level, size the test from the baseline rate, the minimum detectable effect, alpha and power, and run at least two full weeks. Check for sample ratio mismatch, analyze at the user level with a confidence interval, and decide based on both statistical and practical significance and the guardrails.

</details>

## 3. Statistical vs. practical significance

**🗣 Say it out loud:** Why doesn't "p < 0.05" mean "worth launching"?

<details>
<summary><b>Reference answer</b></summary>

(1) Statistically significant ≠ practically important: with a huge sample, trivial differences become significant. Look at the effect size and confidence interval and compare them with the minimum meaningful lift set in advance. (2) Guardrail metrics may get worse. (3) Launching has costs: engineering maintenance, complexity, effects on other teams. (4) Short-term effects are not long-term effects (novelty effect). (5) The p-value may have been picked out after repeated peeking or from many metrics.

Conversely, p > 0.05 doesn't mean "no effect" either; the test may simply be underpowered. In that case check whether the confidence interval rules out a meaningful effect.

**Short version:** Statistical significance isn't practical significance: with huge samples tiny effects are significant. I'd look at the effect size and its interval against the minimum worthwhile lift, check guardrails, costs, and whether the result could be a novelty effect or a product of peeking.

</details>

## 4. Sample size and power

The sample size needed per group (comparing two means, two-sided test) is approximately:

$$n \approx \frac{2\,\sigma^2\,(z_{1-\alpha/2} + z_{1-\beta})^2}{\delta^2}$$

where δ is the minimum detectable effect (MDE). With α = 0.05 and power = 0.8, $(1.96 + 0.84)^2 \approx 7.85$, and the common shortcut is $n \approx 16\,\sigma^2/\delta^2$.

**The single most important rule: n is inversely proportional to δ squared.** Halve the effect you want to detect and you need 4 times the sample.

**🗣 Say it out loud:** How do you decide how long to run an experiment? What if there isn't enough traffic?

<details>
<summary><b>Reference answer</b></summary>

**Duration** = required sample size ÷ users entering the experiment per day, and in addition: at least one to two full weeks (covering weekdays and weekends); long enough for novelty effects to fade; long enough to cover the metric's own observation window (for 7-day retention you need to wait 7 more days).

**When traffic is short:** (1) raise the MDE (only detect larger effects); (2) switch to a more sensitive, lower-variance metric (e.g. conversion rate instead of revenue); (3) **reduce variance**: covariate adjustment with pre-experiment data (CUPED), stratification, capping heavy-tailed metrics; (4) allocate more traffic to the experiment; (5) run longer; (6) accept lower power and treat the result as directional evidence.

**Short version:** Duration is the required sample size divided by daily eligible traffic, with a floor of one or two full weeks to cover weekly cycles and novelty effects. If traffic is short, I'd raise the MDE, pick a less noisy metric, or reduce variance with pre-experiment covariates such as CUPED.

</details>

## 5. Peeking

If you check the p-value every day and stop the experiment to declare victory as soon as it drops below 0.05, the Type I error rate ends up far above 5%. Why: under H₀ the p-value fluctuates randomly, and the more often you look, the more chances it has to dip below 0.05 at some point.

**🗣 Say it out loud:** Can you look at the data while an experiment is running? How do you do sequential monitoring correctly?

<details>
<summary><b>Reference answer</b></summary>

You can look; the problem is not "looking" but "deciding when to stop based on what you see while still using a fixed-sample-size test". The inference method has to match the stopping rule.

Approaches: (1) fix the sample size and duration in advance, do the formal analysis at the end, and in the meantime only monitor guardrails and data quality; (2) use a sequential testing method: group-sequential designs with alpha-spending functions (O'Brien–Fleming, Pocock), mSPRT / always-valid p-values; (3) Bayesian methods with decision rules set in advance.

**Stopping early** for safety reasons (the treatment is clearly harmful) is a different matter and can be done at any time.

**Short version:** Looking is fine; stopping based on what you see while using a fixed-horizon test inflates the false-positive rate. Either fix the sample size in advance, or use a sequential method such as group-sequential boundaries with alpha spending, or always-valid p-values.

</details>

## 6. Sample ratio mismatch (SRM)

You planned a 50/50 split, but the actual split is 50.9/49.1 with a large sample. Use a chi-square goodness-of-fit test; a tiny p-value (the usual threshold is 0.001) means **something went wrong with assignment or data logging**, the experiment's results can't be trusted, and you should find the cause first instead of analyzing the metrics.

Common causes: a bug in the randomization code, crashes or slow loading in one group causing lost logs, bot filtering that treats the groups asymmetrically, an experiment trigger condition related to the treatment itself, redirects that drop users.

## 7. Other common pitfalls

**🗣 Say it out loud:** Why randomize by user rather than by page view or session? What do you need to watch for in the analysis?

<details>
<summary><b>Reference answer</b></summary>

(1) **Consistent experience:** the same user shouldn't flip between the new and old versions; (2) **independence:** multiple page views from the same user are correlated, not independent samples; (3) many metrics (retention, revenue per user) are user-level to begin with.

In the analysis, **the unit of analysis must match the unit of randomization**. If you randomize by user but compute standard errors treating each page view as an independent sample, you badly underestimate the variance and get lots of false positives. For ratio metrics like click-through rate (total clicks / total views), both numerator and denominator are sums over users, so compute the standard error with the delta method or a user-level bootstrap.

**Short version:** Users should get a consistent experience, and a user's events are correlated, so the user is the independent unit. The analysis unit must match the randomization unit; treating page views as independent understates the variance. For ratio metrics like CTR I'd use the delta method or a user-level bootstrap.

</details>

**🗣 Say it out loud:** You looked at 30 metrics and 2 are significant. How do you interpret that?

<details>
<summary><b>Reference answer</b></summary>

If there are actually no effects, you'd expect 1.5 of 30 metrics to be "significant" at the 0.05 level, so 2 significant ones may well be chance. The right approach: pre-specify one primary metric for the decision; treat the others as exploratory, with Bonferroni or BH correction, or clearly label them as "hypotheses to be confirmed by a follow-up experiment". The same goes for hunting for significant results across user segments after the fact.

**Short version:** With 30 metrics you'd expect one or two false positives at the 5% level, so two hits is consistent with noise. Decisions should rest on a pre-specified primary metric; the rest are exploratory and need multiplicity correction or a follow-up test.

</details>

**🗣 Say it out loud:** When does a standard A/B test break down or become impossible? What are the alternatives?

<details>
<summary><b>Reference answer</b></summary>

**Interference / network effects:** one user's treatment affects other users, as in social products and two-sided marketplaces (giving coupons to riders in the treatment group takes drivers away from the control group). Remedy: randomize by cluster, region or time period (cluster / geo / switchback experiments).

**Novelty and learning effects:** users click more because something is new, or need time to adapt. Remedy: run longer, look at new vs. existing users separately, and look at how the effect changes over time.

**Randomization not possible:** legal or ethical constraints, changes launched to everyone, pricing. Remedy: quasi-experimental methods such as difference-in-differences (DiD), synthetic control, regression discontinuity, instrumental variables, matching, stating clearly the assumptions each relies on.

**Short version:** Network effects or marketplace interference violate independence between units, so I'd randomize by cluster, geography, or time with a switchback design. Novelty effects call for longer tests. If randomization is impossible, quasi-experimental methods like difference-in-differences or synthetic control are the fallback.

</details>

**🗣 Say it out loud:** What is CUPED? Why is it useful?

<details>
<summary><b>Reference answer</b></summary>

CUPED uses data from **before the experiment started** (usually the same metric's pre-experiment value) as a covariate to reduce the metric's variance. The adjusted metric is $Y - \theta(X - \bar X)$, where $\theta = \mathrm{Cov}(X, Y)/\mathrm{Var}(X)$. Because X is measured before the experiment, it is independent of assignment, so the adjustment introduces no bias; the variance shrinks by a factor of $(1 - \rho^2)$, where ρ is the correlation between the pre- and in-experiment metric. A correlation of 0.7 cuts the variance roughly in half, like doubling the sample size for free. In essence it is regression adjustment (ANCOVA).

**Short version:** CUPED uses pre-experiment data, typically the same metric before the test, as a covariate to remove variance. Because the covariate is independent of assignment it doesn't bias the estimate, and variance drops by a factor of one minus the squared correlation, which is like getting extra sample size for free.

</details>

## Checkpoint

You can design a complete A/B test (10 steps) and answer questions on: power and sample size, p-values vs. launch decisions, peeking, SRM, the unit of analysis, multiple metrics, network effects.

On to **09b**.